<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: The average hides the people

**[The average hides the people](https://learning.nextia-ai.com/courses/experiments/m07/the-average-hides-the-people/)** · Experiments and Causal Impact: Did Our Change Work? · Module 7, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** look under the average effect of the reminder email, find a group that it helps a lot and a group that it harms, and meet the four groups of uplift modelling.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Segments planned in advance, from [Segments and different effects](https://learning.nextia-ai.com/courses/experiments/m03/segments-and-different-effects/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m07/the-average-hides-the-people/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M07-L01-the-average-hides-the-people/the-average-hides-the-people-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "email/customers.csv.gz": "e16d24a15f1260876a6d73117b24604495f4af0abc855978b7e5b2828ecd3d3c",
    "email/truth.json": "80bc73295fefa15bf0567269825b5e71402abdc98180deaa302e3a05396b9fd3"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The average effect

The reminder email test again: B got the email. One number for everyone:

In [ ]:
email = pd.read_csv("data/email/customers.csv.gz")

def effect(df):
    p = df.groupby("variant")["ordered"].mean()
    n = df.groupby("variant")["ordered"].count()
    se = np.sqrt(p["A"] * (1 - p["A"]) / n["A"] + p["B"] * (1 - p["B"]) / n["B"])
    return p["B"] - p["A"], se

d, se = effect(email)
print(f"everyone: {d:+.4f} (± {1.96 * se:.4f})")

> **Check.** You should see `everyone: +0.0147 (± 0.0047)`.

## 2. New customers and heavy buyers

Rafael planned two segments before the test: new customers, and heavy buyers (7 or more orders in a year).

> **Predict.** Will the email help heavy buyers more or less than new customers?

In [ ]:
for name, part in [("new customers", email[email.new_customer == 1]),
                   ("returning customers", email[email.new_customer == 0]),
                   ("heavy buyers (7+ orders)", email[email.orders_12m >= 7])]:
    d, se = effect(part)
    print(f"{name:<26} {len(part):>6} customers: {d:+.4f} (± {1.96 * se:.4f})")

> **Check.** You should see new customers +0.0774 and heavy buyers -0.0626.

The email helps new customers a lot, and it **lowers** the orders of heavy buyers: they did not need a reminder, and some unsubscribed. The average (+0.0147) hides both.

## 3. The four groups

Each customer has two possible outcomes: with the email and without it. Four groups follow: **persuadables** order only with the email, **sure things** order anyway, **lost causes** never order, and **do-not-disturbs** order only without it. A real test never sees both outcomes of one person. The simulator does:

In [ ]:
groups = truth("email")["groups"]
for g, k in groups.items():
    print(f"{g:<15} {k:>6} ({k / sum(groups.values()):.1%})")

> **Check.** You should see persuadables 1567 (2.6%) and do-not-disturbs 853 (1.4%).

The true average effect is persuadables minus do-not-disturbs, divided by everyone: (1,567 − 853) ÷ 60,000 = 0.0119. The test estimated +0.0147: close, with chance in it. To send fewer, better emails, we need to find the persuadables and avoid the do-not-disturbs (Module 7, lessons 2 and 3).

## 4. Your turn: customers who open emails

> **Your turn.** Compute the effect (B − A in the share who ordered) among customers with `email_open_rate` of 0.5 or more. Put it in `open_effect`, rounded to 4 decimals.

In [ ]:
open_effect = round(float(effect(email[email.email_open_rate >= 0.5])[0]), 4)
print(open_effect)

In [ ]:
expect('open_effect', open_effect, 0.0352)

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m07/the-average-hides-the-people/) has the 2 × 2 picture of the four groups and the knowledge checks that count toward your certificate. Next, uplift models score each customer instead of a few segments.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Uplift models](https://learning.nextia-ai.com/courses/experiments/m07/uplift-models/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m07/the-average-hides-the-people/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.